In [3]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/gensim312/bin/python


In [2]:
import numpy as np
import scipy
import sklearn
import gensim

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import Word2Vec

In [3]:
texts = [
    "I love this movie",
    "This movie is amazing",
    "I really enjoyed this film",
    "This is a wonderful movie",
    "This movie is great",
    "I love this wonderful film",

    "I hate this movie",
    "This movie is terrible",
    "I disliked this film",
    "This is a boring movie",
    "This movie is bad",
    "I hate this boring film"
]

labels = [
    "positive", "positive", "positive",
    "positive", "positive", "positive",
    "negative", "negative", "negative",
    "negative", "negative", "negative"
]

print("Total texts:", len(texts))
print("Total labels:", len(labels))

Total texts: 12
Total labels: 12


## TRAIN / TEST SPLIT

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 9
Testing samples: 3


## BAG OF WORDS

In [5]:
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

bow_model = LogisticRegression(random_state=42)
bow_model.fit(X_train_bow, y_train)

bow_pred = bow_model.predict(X_test_bow)

print("BAG OF WORDS")
print("Vocabulary:", bow_vectorizer.get_feature_names_out())
print("Accuracy:", accuracy_score(y_test, bow_pred))

BAG OF WORDS
Vocabulary: ['amazing' 'boring' 'disliked' 'enjoyed' 'film' 'hate' 'is' 'love' 'movie'
 'really' 'terrible' 'this' 'wonderful']
Accuracy: 0.3333333333333333


## N-GRAMS

In [6]:
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))

X_train_ngram = ngram_vectorizer.fit_transform(X_train)
X_test_ngram = ngram_vectorizer.transform(X_test)

ngram_model = LogisticRegression(random_state=42)
ngram_model.fit(X_train_ngram, y_train)

ngram_pred = ngram_model.predict(X_test_ngram)

print("N-GRAMS")
print("Accuracy:", accuracy_score(y_test, ngram_pred))

N-GRAMS
Accuracy: 0.3333333333333333


## TF-IDF

In [7]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_model = LogisticRegression(random_state=42)
tfidf_model.fit(X_train_tfidf, y_train)

tfidf_pred = tfidf_model.predict(X_test_tfidf)

print("TF-IDF")
print("Accuracy:", accuracy_score(y_test, tfidf_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    tfidf_pred,
    zero_division=0
))

TF-IDF
Accuracy: 0.3333333333333333

Classification Report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         2
    positive       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3



## PREDICT NEW TEXT USING TF-IDF

In [8]:
new_text = ["I really love this wonderful movie"]

new_text_vector = tfidf_vectorizer.transform(new_text)

prediction = tfidf_model.predict(new_text_vector)

print("TF-IDF PREDICTION")
print("Text:", new_text[0])
print("Predicted sentiment:", prediction[0])

TF-IDF PREDICTION
Text: I really love this wonderful movie
Predicted sentiment: positive


## WORD2VEC MODEL

In [11]:
tokenized_texts = [
    text.lower().split()
    for text in texts
]

w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=50,
    window=2,
    min_count=1,
    workers=1,
    seed=42,
    epochs=100
)

print("WORD2VEC")
print("Python-independent example: movie vector")
print(w2v_model.wv["movie"])

WORD2VEC
Python-independent example: movie vector
[ 0.01074516 -0.01203906 -0.00606514 -0.00183739  0.0001529  -0.01796537
  0.00098503 -0.01333717  0.00846954  0.0075486   0.01745187  0.01111402
 -0.00450952  0.02021968 -0.00280797 -0.00556256  0.01551142 -0.0072257
 -0.01791224 -0.00096813  0.01083125 -0.01234225 -0.00182235 -0.01434206
  0.0096596  -0.00140554 -0.00635025 -0.01118568  0.00232197  0.00640608
  0.01799267 -0.00241177 -0.01480631  0.01395085  0.00541557  0.00794697
 -0.01634738 -0.00753879  0.01038188  0.01371184 -0.00203487  0.01262555
  0.01442544 -0.00490223  0.01685291 -0.00865721 -0.01051806  0.00706983
  0.0057517  -0.01603494]


## DOCUMENT VECTORS WITHOUT def

In [13]:
# Each document vector is the average of its word vectors.

X_w2v = []

for words in tokenized_texts:

    vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    if len(vectors) > 0:
        doc_vector = np.mean(vectors, axis=0)
    else:
        doc_vector = np.zeros(w2v_model.vector_size)

    X_w2v.append(doc_vector)

X_w2v = np.array(X_w2v)

print("DOCUMENT VECTORS")
print("Shape:", X_w2v.shape)

DOCUMENT VECTORS
Shape: (12, 50)


## WORD2VEC + CLASSIFIER

In [14]:
w2v_classifier = LogisticRegression(random_state=42)

w2v_classifier.fit(X_w2v, labels)

new_words = "I love this amazing movie".lower().split()

new_vectors = [
    w2v_model.wv[word]
    for word in new_words
    if word in w2v_model.wv
]

if len(new_vectors) > 0:
    new_vector = np.mean(new_vectors, axis=0)
else:
    new_vector = np.zeros(w2v_model.vector_size)

prediction = w2v_classifier.predict([new_vector])

print("WORD2VEC PREDICTION")
print(prediction[0])

WORD2VEC PREDICTION
positive


## HEURISTIC CLASSIFIER WITHOUT def

In [15]:
positive_words = [
    "love", "amazing", "wonderful",
    "great", "enjoyed"
]

negative_words = [
    "hate", "terrible", "boring",
    "bad", "disliked"
]

text = "I love this amazing movie".lower()

positive_count = sum(
    word in text.split()
    for word in positive_words
)

negative_count = sum(
    word in text.split()
    for word in negative_words
)

if positive_count > negative_count:
    heuristic_prediction = "positive"

elif negative_count > positive_count:
    heuristic_prediction = "negative"

else:
    heuristic_prediction = "unknown"

print("HEURISTIC PREDICTION")
print(heuristic_prediction)

HEURISTIC PREDICTION
positive


## COMPARE CLASSIFICATION METHODS

In [16]:
print("MODEL ACCURACY COMPARISON")

print("BoW:", accuracy_score(y_test, bow_pred))
print("N-Gram:", accuracy_score(y_test, ngram_pred))
print("TF-IDF:", accuracy_score(y_test, tfidf_pred))

MODEL ACCURACY COMPARISON
BoW: 0.3333333333333333
N-Gram: 0.3333333333333333
TF-IDF: 0.3333333333333333
